# El Niño 2026 en la costa norte del Perú con NISAR (banda L)
### Procesamiento de escenas GCOV, GUNW y SME2 para Tumbes y Piura, cruzado con noticias

El código vive en el paquete `nisar_cn/` (un archivo por etapa) y este notebook lo importa y explica. Sigue la estructura de `pipeline/fetch_nisar.py` del proyecto *nsac-project-main* del equipo: búsqueda, geometría única, lectura de la ventana de la zona, clasificación y manifiesto para la web. Sobre esa base agrega:

1. **Zonas y eventos referenciados.** Cada zona entra porque una noticia abierta y citada documenta un daño en una fecha concreta.
2. **Pares antes/después automáticos** para cada evento, con la misma geometría de adquisición.
3. **La física que pide el reto:** sensibilidad a la **rugosidad** y a la **constante dieléctrica** con el modelo de Dubois (HH) y la relación de Topp (ε ↔ humedad). Incluye un mapa de Δε por detección de cambios.
4. **Indicadores adicionales:** Δσ⁰ (dB), cociente HH/HV, coherencia (GUNW) y humedad del suelo (SME2).
5. **Tabla para el storytelling** y un manifiesto compatible con la app web del equipo.

**Modos de ejecución**
- `MODO = "DEMO"`: no descarga nada. Usa el catálogo real de escenas que consulté en el CMR el 14 nov 2026 y genera archivos HDF5 **sintéticos** con la misma estructura que un GCOV, para probar el código de punta a punta. **Ninguna cifra del modo DEMO es un resultado científico.**
- `MODO = "REAL"`: consulta el CMR en vivo, descarga las escenas con tu token de Earthdata y procesa los datos reales.

**Convención de fuentes (regla del proyecto):** cada afirmación lleva una **caja Fuente** con estatus:
**[V]** verificado en una fuente abierta en la sesión de preparación (14 nov 2026) · **[A]** deducción propia a partir de una fuente citada (se muestra el razonamiento) · **[?]** hueco: falta referencia y se pide al equipo.

## Mapa del notebook: qué se hace, con qué datos y cómo se comprueba

| Paso | Qué hace | Datos | Sección |
|---|---|---|---|
| 1 | Buscar todas las escenas NISAR de cada zona (ago–nov 2026) | NASA CMR (R14) | 3 |
| 2 | Leer **solo** la ventana de cada zona por streaming, sin bajar los 6–7 GB | NISAR L2 GCOV (R2) | 4–5 |
| 3 | **Limpiar**: máscara de validez, relleno, moteado (Lee con los looks reales), ángulo de incidencia por píxel | capas `mask`, `numberOfLooks`, cubo `incidenceAngle` (R16) | 5.2 |
| 4 | Medir: agua, vegetación inundada, ciudad, Δσ⁰, Δε′ (humedad), HV (vegetación) | `HHHH`, `HVHV`, `rtcGammaToSigmaFactor` | 6 |
| 5 | Serie temporal por zona y órbita: agua permanente, control de calidad, salto de madurez | todas las fechas | 7 |
| 6 | Otros productos NISAR: humedad del suelo SME2, coherencia GUNW | R3, R4 | 8 |
| 7 | **Corroborar** con fuentes independientes: lluvia satelital IMERG y reanálisis, estaciones Senamhi, caudal del río Tumbes y sus umbrales, Carta Internacional, noticias | R18–R22, N1–N17 | 9 |
| 8 | Exportar tabla, manifiesto y mapa interactivo | — | 10 |
| 9 | Huecos que siguen abiertos | — | 11 |

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
pasos = [("NASA CMR\n(búsqueda)", "#4c78a8"), ("GCOV por streaming\n(ventana)", "#4c78a8"),
         ("Limpieza\nmask · Lee · θ", "#f58518"), ("Indicadores\nagua · ε′ · HV", "#54a24b"),
         ("Serie temporal\nperm. · QA · madurez", "#54a24b"), ("SME2 · GUNW", "#54a24b"),
         ("Corroboración\nIMERG · caudal · noticias", "#e45756"), ("Mapa y tabla", "#72b7b2")]
fig, ax = plt.subplots(figsize=(15, 2.2)); ax.axis("off")
for i, (t, c) in enumerate(pasos):
    ax.add_patch(FancyBboxPatch((i * 1.9, 0), 1.6, 1, boxstyle="round,pad=0.05", fc=c, ec="none", alpha=.9))
    ax.text(i * 1.9 + .8, .5, t, ha="center", va="center", color="white", fontsize=9, weight="bold")
    if i < len(pasos) - 1:
        ax.annotate("", xy=(i * 1.9 + 1.85, .5), xytext=(i * 1.9 + 1.62, .5), arrowprops=dict(arrowstyle="->", lw=2))
ax.set_xlim(-.1, len(pasos) * 1.9); ax.set_ylim(-.1, 1.1); plt.title("Flujo de procesamiento"); plt.show()

## 0. Referencias

| ID | Referencia | Qué respalda | Estatus |
|---|---|---|---|
| R1 | NASA Space Apps Challenge 2026, reto *Dancing with the SARs*, pestaña *Details*, sección *Background*. https://www.spaceappschallenge.org/2026/challenges/dancing-with-the-sars/?tab=details | El radar es más sensible a la rugosidad y a la constante dieléctrica; el agua tiene constante dieléctrica alta | [V] |
| R2 | ASF DAAC, *NISAR Data User Guide*, página «Geocoded Polarimetric Covariance (GCOV)». https://nisar-docs.asf.alaska.edu/gcov/ (cita a Shiroma, G. H. X. (2026), *Level-2 Geocoded Polarimetric Covariance*, JPL D-102274 Rev. F) | Rutas HDF5 (`HHHH`, `HVHV`, `numberOfLooks`, `mask`, `rtcGammaToSigmaFactor`), γ⁰ con RTC, píxel de 10/20 m según ancho de banda, convención de nombres (`MODE`, `POLE`, `CRID`) | [V] |
| R3 | ASF DAAC, *NISAR Data User Guide*, «Data Products Overview». https://nisar-docs.asf.alaska.edu/products-overview/ | Madurez BETA/PROVISIONAL/VALIDATED, precaución al mezclarlas, VALIDATED desde 21 oct 2026, SME2 a 200 m con 3 algoritmos (DSG, PMI, TSR) | [V] |
| R4 | ASF DAAC, *NISAR Data User Guide*, «GUNW». https://nisar-docs.asf.alaska.edu/gunw/ | GUNW: solo co-pol (HH o VV), pares de 12 días, coherencia a 20 y 80 m | [V] |
| R5 | ASF DAAC, *NISAR Data User Guide*, «About NISAR», tabla 1. https://nisar-docs.asf.alaska.edu/nisar-intro/ | Frecuencia central de la banda A según el modo (40+5 MHz → 1239 MHz; 20+5 MHz → 1229 MHz) | [V] |
| R6 | NASA Earthdata, «Synthetic Aperture Radar (SAR)». https://www.earthdata.nasa.gov/learn/earth-observation-data-basics/sar | Mecanismos: superficie rugosa (sensible a VV), volumen (sensible a HV/VH), rebote doble por troncos o vegetación inundada (sensible a HH); la banda L (~23 cm) penetra el dosel | [V] |
| R7 | Meyer, F. J. (2023). *GEOS 657 Microwave Remote Sensing*, Lecture 17 «Surface Water Mapping from SAR», University of Alaska Fairbanks, diapositivas 5, 6, 7 y 10. https://radar.community.uaf.edu/wp-content/uploads/sites/667/2023/04/2023-Lectures17-SurfaceWaterMapping-handouts.pdf | El agua en calma da reflexión especular y retrodispersión baja; la tierra es más rugosa; rebote doble bajo vegetación inundada; σ⁰ sube con la humedad del suelo en terreno abierto | [V] |
| R8 | Dubois, P., van Zyl, J., Engman, T. (1995). «Measuring soil moisture with imaging radars». *IEEE Trans. Geosci. Remote Sens.* 33(4), 915–926. doi:10.1109/36.406677 | Modelo semiempírico σ⁰_HH(ε, ks, θ, λ) y límites de validez. **El artículo original no se abrió**: la ecuación se tomó de la implementación SENSE (`sense.surface.dubois95`, que cita a Ulaby 2014, cap. 10.6, ec. 10.41) y los límites de arXiv:2412.11874 | [V] (fuente secundaria) |
| R9 | Topp, G. C., Davis, J. L., Annan, A. P. (1980). «Electromagnetic determination of soil water content: measurements in coaxial transmission lines». *Water Resour. Res.* 16, 574–582 | ε = 3.03 + 9.3 m_v + 146.0 m_v² − 76.7 m_v³. **Original no abierto**: tomada de arXiv:1006.4307 | [V] (fuente secundaria) |
| R10 | Otsu, N. (1979). «A Threshold Selection Method from Gray-Level Histograms». *IEEE Trans. Syst. Man Cybern.* 9(1), 62–66. doi:10.1109/TSMC.1979.4310076 | Umbral de Otsu (el que usa el código del equipo). Metadatos vistos en resultados de búsqueda; **artículo no abierto** | [?] |
| R11 | NASA Earthdata, alerta «Permanent NISAR Instrument Data Gap» (28 ago 2026). https://www.earthdata.nasa.gov/data/alerts-outages/permanent-nisar-instrument-data-gap | Corte de datos de 2026-07-27 22:03:25 a 2026-08-10 00:55:27 UTC | [V] |
| R12 | NASA Earthdata, alerta «NISAR provisional data product production paused». https://www.earthdata.nasa.gov/data/alerts-outages/nisar-provisional-data-product-production-paused | Producción PROVISIONAL detenida el 2026-10-23 23:53 UTC; reanudación con atrasado después del 6 nov 2026 | [V] |
| R13 | WMO OSCAR, instrumento «SAR-L (NISAR)». https://space.oscar.wmo.int/instruments/view/sar_l_nisar | Ángulo de incidencia 33–47°, banda 1.215–1.3 GHz, modos de polarización | [V] |
| R14 | NASA CMR (Common Metadata Repository), consultas del 14 nov 2026. https://cmr.earthdata.nasa.gov/search/ | IDs de colección y escenas concretas (fechas, track, frame, enlaces HDF5) | [V] |
| R15 | Código del equipo `nsac-project-main` (`pipeline/fetch_nisar.py`, `README.md`) | Estructura del pipeline, sesión de Earthdata, lectura de la ventana, umbrales de clasificación | [V] (como código) |

**Noticias (todas abiertas el 14 nov 2026)**

| ID | Medio y fecha | Contenido usado |
|---|---|---|
| N1 | América TV, 23 oct 2026. https://www.americatv.com.pe/noticias/amp/actualidad/tumbes-lluvia-mas-ocho-horas-provoca-inundaciones-n523751 | Más de 8 h de lluvia en Tumbes; Zarumilla, Puerto Pizarro y Tumbes ciudad afectados; 52 mm en Puerto Pizarro según Senamhi |
| N2 | Caretas, 20 oct 2026. https://caretas.pe/?p=840084 | Lluvias desde la noche del 19 oct; ~30 cm de agua en el mercado de Chulucanas; también Tambogrande, Castilla, Sullana y Paita; alerta por quebradas en Ayabaca y Huancabamba |
| N3 | Caretas, 3 nov 2026. https://caretas.pe/?p=842604 | Desborde del río Zarumilla la madrugada del 2 nov en Puerto Perú (Aguas Verdes): más de 300 familias; El Tigre 1 021,33 m³/s; El Salto 104,1 mm/24 h |
| N4 | El Búho, 2 nov 2026. https://elbuho.pe/?p=441556 | Río Tumbes 1 021,33 m³/s (El Tigre, 05:00); desborde de la quebrada Marco Felipe (Zarumilla); Corrales y San Juan de la Virgen inundados; 70 % de la lluvia anual en menos de 30 días (aeropuerto) |
| N5 | Andina, 2 nov 2026. https://andina.pe/agencia/noticia-el-nino-tumbes-endures-extraordinary-rainfall-of-up-to-110-liters-per-square-meter-1098315.aspx | Zarumilla, la provincia más afectada; El Salto 110,3 mm el 2 nov y 102 mm el 27 oct; aviso rojo para Zarumilla, Sullana y parte de Ayabaca |
| N6 | El Peruano, 2 nov 2026. https://elperuano.pe/noticia/308191-fuerzas-armadas-activan-puente-terrestre-para-evitar-que-morropon-quede-aislada-por-el-nino | Quebrada Serrán activada en Salitral (Morropón); drenaje en Las Azucenas y en la av. Sánchez Cerro (Piura) |

> **Nota (actualizada el 15 nov 2026):** N3 da 104,1 mm «en 24 horas» para El Salto y N5 da 110,3 mm «durante la madrugada» del 2 nov, sin indicar el periodo. Releídas las dos: son ventanas distintas, no un error; se usan ambas (sección 9.1 b).


**Referencias agregadas el 15 nov 2026 (todas abiertas completas)**

| ID | Referencia | Qué respalda | Estatus |
|---|---|---|---|
| R16 | Shiroma, G. H. X. (2024). *NASA SDS Product Specification, Level-2 Geocoded Polarimetric Covariance (L2 GCOV)*, JPL D-102274 Rev D, v1.1.2. https://nisar.asf.earthdatacloud.nasa.gov/NISAR-SAMPLE-DATA/DOCS/NISAR_D-102274_RevD_NASA_SDS_Product_Specification_L2_GCOV_w-sigs.pdf | `mask`: 0 inválido o parcialmente enfocado, 1–5 válido (número de sub-swath), 255 relleno (sec. 4.3.3, p. 20; tabla 5-3, p. 29). `HHHH`: unidades «1», `_FillValue` nan, `valid_min` 0 (tabla 5-3, p. 29). Cubo `metadata/radarGrid/incidenceAngle` con `xCoordinates`, `yCoordinates`, `heightAboveEllipsoid` (sec. 3.2.4, pp. 12–13; tabla 5-1, p. 25) | [V] (Rev D; la versión de producción puede diferir) |
| R17 | ASF DAAC, *NISAR Data User Guide*, «Product Known Issues» (actualizada 27 feb 2026). https://nisar-docs.asf.alaska.edu/product-known-issues | La máscara de datos válidos está desalineada con los datos y deja artefactos de borde en productos L2; bandas radiométricas por calibración incompleta | [V] |
| R18 | Breña Aliaga, J. C., Vidal, J., Felipe, O., Bourrel, L., Rau, P., Lavado-Casimiro, W. (2026). «Estimation of Flood Thresholds for Hydrological Warning Purposes Using Sentinel-1 SAR Imagery-Based Modeling in the Tumbes River Basin (PERU)». *Remote Sensing* 18(10), 1493. doi:10.3390/rs18101493 | Umbral de activación de impactos 743,49 m³/s y pico extremo 1 629,02 m³/s en el río Tumbes; Bmax Otsu + HAND; serie temporal para separar agua permanente; filtro Lee-Sigma y RTC sobre Sentinel-1 (banda C, VV) | [V] |
| R19 | NASA Earthdata, ficha «GPM IMERG Late Precipitation L3 1 day 0.1° V07 (GPM_3IMERGDL)», concept ID C2723754859-GES_DISC. https://www.earthdata.nasa.gov/data/catalog/ges-disc-gpm-3imergdl-07 · cita: Huffman et al. (2023), doi:10.5067/GPM/IMERGDL/DAY/07 | Lluvia diaria 0,1°, variable `precipitation` (mm/día, relleno −9999,9), ~14 h de retraso | [V] |
| R20 | Carta Internacional «Espacio y Grandes Catástrofes», activación 1058 «Flood in Peru» (3 nov 2026). https://disasterscharter.org/activations/flood-in-peru-activation-1058- | Activación pedida por INDECI en nombre de CONIDA por el desborde del río Zarumilla en Puerto Perú; sin productos publicados aún | [V] |
| R21 | Open-Meteo, «Historical Weather API». https://open-meteo.com/en/docs/historical-weather-api | Endpoint `/v1/archive`, `daily=precipitation_sum`; reanálisis ERA5 / ERA5-Land / ECMWF IFS 9 km; uso no comercial | [V] |
| R22 | Senamhi, «Descarga de datos Meteorológicos». https://www.senamhi.gob.pe/site/descarga-datos/ | Portal de estaciones. El mapa https://www.senamhi.gob.pe/mapas/mapa-estaciones-2/ da código y coordenadas (El Salto 103041: −3,4517, −80,2802; Cabo Inga 103043: −3,9759, −80,4018; Puerto Pizarro 103038: −3,5040, −80,3954). Las tablas de datos de cada estación están detrás de un CAPTCHA (Cloudflare Turnstile) | [V] coordenadas (15 nov 2026) · [?] datos en bruto |
| R23 | Climate Hazards Center (UCSB), «Information about CHIRPS Daily Precipitation» (readme de CHIRPS v3 diario). https://data.chc.ucsb.edu/products/CHIRPS/v3.0/daily/prelim/readme.txt · archivo `chirps-v3.0_prelim.sat.2026.days_p05.nc` | Diario «sat» = total pentadal CHIRPS3 (mezclado con estaciones) repartido por día con NASA IMERG Late V07; días en UTC; malla 0,05°; el diario no es un producto diario independiente, conviene usar acumulados | [V] (leído el 15 nov 2026) |
| R24 | Copernicus Emergency Management Service, Rapid Mapping, API pública `public-activations-info`. https://rapidmapping.emergency.copernicus.eu/backend/dashboard-api/public-activations-info/ | 268 activaciones, la última EMSR935 (19 oct 2026, España); ninguna para Perú (consulta del 15 nov 2026) | [V] |
| R25 | Respuesta de GES DISC a la llave de este proyecto (15 nov 2026): `403 {"error_description":"EULA Acceptance Failure", "resolution_url":"https://urs.earthdata.nasa.gov/approve_app?..."}` · NASA Earthdata Login, «What is application authorization?» https://urs.earthdata.nasa.gov/documentation/for_users/what_is_app_auth | Para descargar de GES DISC hay que autorizar la aplicación y aceptar su EULA | [V] respuesta observada · la página de documentación solo la vi en resultados de búsqueda |
| R26 | WWRP/WGNE Joint Working Group on Forecast Verification Research, «Forecast verification – methods, issues and FAQ». https://jwgfvr.github.io/forecastverification | Tabla de contingencia, POD, FAR, CSI (threat score) y sesgo de frecuencia | [V] |

**Noticias N7–N17:** ver `config.CRONICA` y `config.CONTEXTO`; todas abiertas completas el 14 nov 2026, con medio, fecha y resumen propio.

## 1. Configuración

Todo lo configurable está en `nisar_cn/config.py`. Cada constante lleva allí su fuente.

In [ ]:
MODO = "DEMO"          # datos SINTÉTICOS para probar el código; la versión REAL está en 1_jupyter/

import sys
from pathlib import Path
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "nisar_cn").is_dir())   # carpeta del proyecto
sys.path.insert(0, str(RAIZ))

import datetime as dt
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from nisar_cn import config, fisica, busqueda, lectura, indicadores, exportar, pipeline, limpieza, corroboracion
CACHE, SALIDAS = RAIZ / "cache", RAIZ / "salidas"
print("Modo:", MODO, "| raíz del proyecto:", RAIZ)

### 1.1 Credenciales desde `.env`

El código busca un archivo `.env` en la carpeta del proyecto (`NISAR_ElNino_2026`) y hasta tres carpetas más arriba; si no lo encuentra, usa el de `Descargas\NassaSpaceAppChallenge`. Lo más limpio es copiar tu `.env` dentro de `NISAR_ElNino_2026`. Acepta una llave (`EARTHDATA_TOKEN=...` u otros nombres habituales, ver `nisar_cn/acceso.py`) o el par `EARTHDATA_USER` / `EARTHDATA_PASS`.

**Nunca se imprimen los valores, solo los nombres de las variables.** El `.gitignore` excluye `.env` para que la llave no termine en GitHub.

En modo REAL la celda siguiente también **prueba el acceso** leyendo los primeros 8 bytes de una escena real (24 oct 2026, Tumbes). No descarga el archivo de 6,8 GB. Si las credenciales sirven, esos bytes son la firma de un archivo HDF5.

> **Fuente** — Streaming con credenciales de Earthdata: ASF, *NISAR Data User Guide*, «Stream Data» · Estatus [V]. Escena y URL: R14 · Estatus [V].
> **[?]** La firma de 8 bytes `\x89HDF\r\n\x1a\n` viene de la especificación del formato HDF5 (The HDF Group, *HDF5 File Format Specification*, «Format Signature»), que no abrí en esta sesión. La comprobé en local con un archivo escrito por `h5py`.

In [ ]:
from nisar_cn import acceso
ruta_env, nombres = acceso.cargar_env()
print(".env:", ruta_env if ruta_env else "NO ENCONTRADO")
print("variables definidas (solo nombres):", nombres)
print("credencial que se usará:", acceso.credenciales())

URL_PRUEBA = ("https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L2_GCOV_V1/"
              "NISAR_L2_PR_GCOV_034_003_A_176_4005_DHDH_A_20261024T110112_20261024T110149_P05036_N_F_J_001/"
              "NISAR_L2_PR_GCOV_034_003_A_176_4005_DHDH_A_20261024T110112_20261024T110149_P05036_N_F_J_001.h5")
if MODO == "REAL":
    try:
        ok, cab = acceso.probar_acceso(URL_PRUEBA)
        print("Acceso a NASA:", "OK — es un HDF5" if ok else f"respuesta inesperada {cab!r} (¿la llave expiró?)")
    except Exception as e:
        print("Acceso a NASA: FALLÓ →", type(e).__name__, str(e)[:300])
        print("Si es 401/403: entra una vez en https://search.asf.alaska.edu con tu cuenta y autoriza la aplicación ASF,"
              " o genera una llave nueva en https://urs.earthdata.nasa.gov (Generate Token).")
else:
    print("DEMO: no se prueba el acceso. Cambia MODO = 'REAL' en la primera celda.")

### 1.2 Bases de datos que usa el proyecto

| Base | Variable | Resolución | Cómo se accede | Papel | Fuente |
|---|---|---|---|---|---|
| NISAR L2 GCOV | γ⁰ HH, HV; factor γ⁰→σ⁰; looks; mask; cubo θ | 10–20 m | CMR + streaming HTTPS con la llave del `.env` | **dato principal** | R2, R16 [V] |
| NISAR L3 SME2 | humedad del suelo | 200 m | igual | humedad medida por NASA (contraste de Δε′) | R3 [V] |
| NISAR L2 GUNW | coherencia 12 días | 80 m | igual | cambio de superficie entre pasadas | R3, R4 [V] |
| NASA GPM IMERG Late | lluvia diaria | 0,1° (~10 km) | CMR + HTTPS, misma llave **y aceptar la EULA de GES DISC** | ¿llovió entre dos pasadas? (satélite) | R19, R25 [V] |
| CHIRPS v3 diario «sat» | lluvia diaria | 0,05° | HTTPS abierto, sin llave; solo se lee la ventana | satélite + estaciones; cubre si IMERG falla | R23 [V] |
| Open-Meteo | lluvia diaria | reanálisis ECMWF | API pública | segunda estimación de lluvia | R21 [V] |
| Estaciones Senamhi (vía prensa) | mm medidos | puntual, con coordenadas oficiales | notas N1, N3, N5, N8–N10, N15; mapa de estaciones (R22) | lluvia observada en tierra | [V] |
| Caudal río Tumbes (El Tigre) | m³/s | puntual | notas N3, N4 | crecida observada | [V] |
| Breña Aliaga et al. (2026) | umbrales de caudal–inundación | cuenca del Tumbes | artículo | ¿el caudal observado implica inundación? | R18 [V] |
| Verificación de pronósticos (JWGFVR) | POD, FAR, CSI | — | web | puntaje de la correspondencia NISAR ↔ tierra | R26 [V] |
| Carta Internacional, activación 1058 | activación oficial | Zarumilla | web | confirmación institucional del evento | R20 [V] |
| Copernicus EMS Rapid Mapping | activaciones | mundial | API pública | ¿hay mapas de Copernicus del evento? | R24 [V] |
| Crónica de noticias | hechos fechados por zona | distrito | 16 notas abiertas | qué pasó y cuándo | N1–N17 [V] |

**Colecciones NISAR**

> **Fuente** — R14 (IDs leídos del CMR) · R3 (significado de cada madurez y precaución al mezclarlas) · R4 (GUNW) · R3 (SME2 sin versión VALIDATED aún) · Estatus [V]

In [ ]:
config.COLECCIONES

**Restricciones temporales del registro NISAR**

> **Fuente** — R11 (corte permanente de datos) · R12 (pausa de la producción PROVISIONAL desde el 23 oct 2026 23:53 UTC, con atrasado después del 6 nov) · R3 (VALIDATED desde el 21 oct 2026) · Estatus [V]
>
> La pausa de R12 explica por qué, al 14 nov 2026, todavía no había escenas de Tumbes posteriores al 2 nov en el CMR (R14).

In [ ]:
config.GAP_START, config.GAP_END, config.PAUSA_PR

**Zonas de estudio.** Cada zona está ligada a la noticia que la justifica. El punto (lat, lon) lo elegí yo para la consulta y la caja es de ±`R_DEG` grados alrededor.

> **Fuente** — N1–N6 (daños y lugares) · R14 (en los puntos de Tumbes, Zarumilla, Puerto Pizarro, Piura, Chulucanas, Salitral, Sullana y Ayabaca la consulta del 14 nov 2026 devolvió escenas GCOV) · Estatus de las coordenadas: **[A]**, puntos elegidos por mí y no tomados de una fuente.
>
> **[?] Hueco:** las coordenadas de los sectores que nombran las noticias (Puerto Perú, Las Azucenas, quebradas Marco Felipe y Serrán) deberían venir de una fuente cartográfica (IGN, INEI u OpenStreetMap). El equipo debe proporcionarla o confirmarla.

In [ ]:
pd.DataFrame([{"id": k, "nombre": z["nombre"], "región": z["region"], "noticias": z["noticias"],
               "lat": z["punto"][0], "lon": z["punto"][1]} for k, z in config.ZONAS.items()])

**Eventos.** Ventana de cada evento según la noticia (hora local Lima, UTC−5). El análisis buscará la última escena antes de `inicio` y la primera después de `fin`, en la misma geometría.

> **Fuente** — N2 (Piura: lluvias desde la noche del 19 oct) · N1 (Tumbes: 23 oct) · N5 (El Salto: 102 mm el 27 oct) · N3, N4 y N5 (1–2 nov) · N6 (Salitral, 2 nov) · Estatus [V]
>
> **[A]** Las horas de inicio y fin son una acotación mía de cada ventana a partir de las fechas que dan las noticias.

In [ ]:
pd.DataFrame([{**{k: v for k, v in e.items() if k != "zonas"}, "zonas": ", ".join(e["zonas"]),
               "noticia": config.NOTICIAS[e["fuente"]]} for e in config.EVENTOS])

**Crónica de noticias y reportes oficiales por zona** (16 notas abiertas completas; `config.CRONICA`). Se usa en la sección 9 y en el mapa.

> **Fuente** — N1–N16 [V] · resúmenes con palabras propias

In [ ]:
pd.DataFrame([{"id": c["id"], "fecha": c["fecha"], "aprox": c.get("fecha_aprox", False), "zonas": ", ".join(c["zonas"]),
               "tipo": c["tipo"], "medio": c["medio"], "resumen": c["texto"]} for c in config.CRONICA])

## 2. Fundamento físico: rugosidad y constante dieléctrica

**Lo que dice el reto.** El texto de fondo del reto afirma que los datos de radar son más sensibles a la rugosidad y a la constante dieléctrica, y que la constante dieléctrica alta del agua resalta cuerpos de agua, inundaciones y humedad del suelo.
> **Fuente** — R1, sección *Background*, 3.er párrafo · Estatus [V]

**Mecanismos de dispersión y polarización**
- Superficie rugosa (suelo desnudo, agua): más sensible a VV.
- Volumen (hojas y ramas): más sensible a la polarización cruzada (HV, VH).
- Rebote doble (edificios, troncos, vegetación inundada): más sensible a HH.
> **Fuente** — R6, sección *Polarization and Scattering Mechanisms* · Estatus [V]

**El agua en calma da reflexión especular y retrodispersión baja; la tierra, más rugosa, da retrodispersión más alta. En terreno abierto σ⁰ sube con la humedad del suelo.**
> **Fuente** — R7, diapositivas 5, 6 y 10 · Estatus [V]

**Consecuencia para NISAR en esta zona.** Las escenas de Tumbes y Piura son `DHDH` (HH + HV) o `SHSH` (solo HH) (R14). El código `DH` significa HH/HV y `SH` solo HH (R2, convención `POLE`). **No hay VV**, que es el canal más sensible a la superficie rugosa (R6).
> **Fuente** — R2, R6, R14 · Estatus [V]

### 2.1 Modelo de Dubois para HH

$$
\sigma^0_{HH} = 10^{-2.75}\,\frac{\cos^{1.5}\theta}{\sin^{5}\theta}\;10^{\,0.028\,\varepsilon'\tan\theta}\;(ks\,\sin\theta)^{1.4}\;\lambda^{0.7}
$$

donde ε' es la parte real de la constante dieléctrica relativa, k = 2π/λ, s es la altura RMS de la rugosidad, θ el ángulo de incidencia y λ la longitud de onda **en cm**.

**Validez:** m_v < 0.35 cm³/cm³, ks < 3, θ > 30°.

> **Fuente** — R8 (Dubois et al., 1995). Ecuación copiada de la implementación SENSE `sense.surface.dubois95._hh`, que cita Ulaby (2014), cap. 10.6. Límites de validez de arXiv:2412.11874, ec. 2–8 · Estatus [V] (fuente secundaria; el artículo original no se abrió)

### 2.2 Relación de Topp (ε ↔ humedad volumétrica)

$$
\varepsilon(m_v) = 3.03 + 9.3\,m_v + 146.0\,m_v^2 - 76.7\,m_v^3
$$

> **Fuente** — R9 (Topp et al., 1980), tomada de arXiv:1006.4307, sección 5 · Estatus [V] (fuente secundaria)

### 2.3 Sensibilidades (deducción a partir de 2.1)

Al tomar 10·log₁₀ de la ecuación de Dubois:

$$
\frac{\partial \sigma^0_{HH,\mathrm{dB}}}{\partial \varepsilon'} = 10\cdot 0.028\,\tan\theta \;\;[\mathrm{dB}],
\qquad
\Delta\sigma^0_{HH,\mathrm{dB}}\big|_{\varepsilon,\theta\ \mathrm{fijos}} = 14\,\log_{10}\!\frac{(ks)_2}{(ks)_1}\;\;[\mathrm{dB}]
$$

Si entre dos fechas **la rugosidad no cambia** (misma ks), el cambio en dB solo depende de ε:

$$
\Delta\varepsilon' = \frac{\Delta\sigma^0_{HH,\mathrm{dB}}}{0.28\,\tan\theta}
$$

> **Fuente** — Derivación algebraica mía sobre la ecuación de R8 · Estatus **[A]**. El supuesto «ks constante entre fechas» es mío. Donde hubo depósito de lodo o cambio de cobertura (quebradas activadas), el supuesto falla, y la detección de cambios mezcla rugosidad y humedad.

**Por qué no se separan rugosidad y humedad con una sola fecha.** Con HH solamente hay una ecuación (2.1) para dos incógnitas (ε', ks). Dubois resuelve ambas con HH **y VV** (R8, ec. de σ⁰_VV), pero estas escenas no tienen VV (R2, R14). Por eso este notebook usa **dos fechas** con ks supuesta constante, y HV solo como indicador cualitativo de dispersión de volumen (R6).
> **Fuente** — R2, R6, R8, R14 + razonamiento mío · Estatus [A]

In [ ]:
lam = fisica.lambda_cm("4005")
print(f"λ banda A, modo 40+5 MHz: {lam:.2f} cm  (R5 + c/f)")
for th in (33, 40, 47):
    print(f"θ={th}°: ∂σdB/∂ε' = {fisica.dsigma_deps_db(np.radians(th)):.3f} dB   [A]")
print("Duplicar ks cambia σ⁰_HH en", round(float(fisica.dsigma_rugosidad_db(1, 2)), 2), "dB   [A]")

In [ ]:
mv = np.linspace(0.02, 0.34, 60)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for ks in (0.3, 0.6, 1.0, 2.0):
    for th, ls in ((33, "-"), (47, "--")):
        ax[0].plot(mv, 10*np.log10(fisica.dubois_hh(fisica.topp_eps(mv), ks, np.radians(th), lam)), ls, label=f"ks={ks}, θ={th}°")
ax[0].set_xlabel("m_v (cm³/cm³)"); ax[0].set_ylabel("σ⁰_HH (dB)"); ax[0].set_title(f"Dubois (R8) + Topp (R9), λ={lam:.1f} cm")
ax[0].legend(fontsize=7, ncol=2)
ks = np.linspace(0.1, 2.9, 60)
for m in (0.05, 0.15, 0.30):
    ax[1].plot(ks, 10*np.log10(fisica.dubois_hh(fisica.topp_eps(m), ks, np.radians(40), lam)), label=f"m_v={m}")
ax[1].set_xlabel("ks"); ax[1].set_ylabel("σ⁰_HH (dB)"); ax[1].set_title("θ = 40°"); ax[1].legend()
plt.tight_layout(); plt.show()
print("Rango de θ (33–47°) según R13. Las curvas son el modelo, no una medición.")

## 3. Búsqueda de escenas

En modo REAL se consulta el CMR por zona, producto y madurez, ordenado por fecha. El nombre de cada escena se interpreta con la convención de R2: `NISAR_IL_PT_PROD_CYL_REL_P_FRM_MODE_POLE_S_Start_End_CRID_A_C_LOC_CTR`.

En modo DEMO se usa un catálogo **fijo** con escenas reales que leí del CMR el 14 nov 2026 (R14) para Tumbes, Zarumilla y Piura, más las fechas que encajan con los eventos.

Código: `nisar_cn/busqueda.py`.

> **Fuente** — R2 (convención de nombres) · R14 (catálogo) · R15 (la búsqueda del equipo usa `asf_search`; aquí se usa el CMR directamente para no depender de una sesión) · Estatus [V]

In [ ]:
CAT = pipeline.construir_catalogo(MODO)
pd.DataFrame([{"zona": z, "escenas": len(v),
               "geometrías": sorted({f"{g['REL']}_{g['P']}_{g['FRM']}" for g in v}),
               "desde": min((g["t"] for g in v), default=None), "hasta": max((g["t"] for g in v), default=None)}
              for z, v in CAT.items()])

In [ ]:
# Línea de tiempo de escenas NISAR por zona, con eventos, corte de datos y pausa de producción
import datetime as dt
fig, ax = plt.subplots(figsize=(15, 4.5))
zs = list(config.ZONAS)
for i, z in enumerate(zs):
    for g in CAT.get(z, []):
        ax.scatter(g["t"], i, s=22, c="#5be3a0" if g.get("madurez") == "VALIDATED" else "#4c78a8", zorder=3)
for e in config.EVENTOS:
    for z in e["zonas"]:
        ax.scatter(e["inicio"], zs.index(z), marker="x", c="r", s=60, zorder=4)
ax.axvspan(config.GAP_START, config.GAP_END, color="grey", alpha=.25, label="corte de datos (R11)")
ax.axvline(config.PAUSA_PR, color="orange", ls="--", label="pausa PROVISIONAL (R12)")
ax.set_yticks(range(len(zs))); ax.set_yticklabels([config.ZONAS[z]["nombre"] for z in zs])
ax.scatter([], [], c="#4c78a8", label="escena PROVISIONAL"); ax.scatter([], [], c="#5be3a0", label="escena VALIDATED")
ax.scatter([], [], marker="x", c="r", label="evento con noticia")
ax.legend(loc="upper left", fontsize=8); ax.set_title("Escenas NISAR GCOV disponibles por zona"); plt.tight_layout(); plt.show()

### 3.1 Pares antes/después por evento

Para cada evento y zona se busca, **en cada geometría (track, dirección, frame) por separado**, la última escena antes del inicio y la primera después del fin. Se descartan las escenas dentro del corte de R11 y se exige el mismo modo de polarización en las dos fechas.

> **Fuente** — R15 (el equipo ya fija una sola geometría para que todos los cuadros sean comparables) · R11 · R3 (mezclar madurez puede producir diferencias de procesamiento: se registra en la columna `mezcla_madurez`) · Estatus [V]

In [ ]:
PARES = pipeline.tabla_pares(CAT)
PARES.drop(columns=["_a", "_d"])

## 4. Acceso a los datos reales por streaming (modo REAL)

Cada GCOV pesa entre 6 y 7 GB (por ejemplo, 6 819 938 304 bytes la escena del 24 oct 2026 de Tumbes, según su ficha CMR). Por eso **no se descargan completos**. La guía oficial indica que los productos NISAR se pueden leer por streaming con `fsspec` sobre HTTPS y credenciales de Earthdata, para tomar solo los bytes necesarios sin bajar el archivo entero.

Código: `nisar_cn/acceso.py`.

Credenciales: variables de entorno `EARTHDATA_USER` y `EARTHDATA_PASS`, como el `aiohttp.BasicAuth` del ejemplo oficial. Como alternativa se puede usar `EARTHDATA_TOKEN`, enviado como cabecera `Authorization: Bearer`, como en el código del equipo.

> **Fuente** — ASF, *NISAR Data User Guide*, sección «Stream Data» (ejemplo con `fsspec.filesystem('https', client_kwargs={'auth': aiohttp.BasicAuth(...)})`, `block_size=16 MB`, `cache_type='background'`) · tamaño del archivo: R14 · cabecera Bearer: R15 · Estatus [V]
>
> **[A]** El ejemplo oficial abre el archivo con `xarray` y el motor `h5netcdf`. Aquí el objeto de `fsspec` se pasa a `h5py.File`, que acepta objetos tipo archivo. Esto no lo probé contra un archivo real.

## 5. Lectura de la ventana de la zona

Se leen solo las filas y columnas de la caja, en la proyección nativa del producto (UTM, R2). Así el área de píxel es exacta (|dx·dy|) y no se reproyecta para el análisis.

Código: `nisar_cn/lectura.py`. Capas que se leen y para qué:

| Capa (R2, R16) | Qué es | Uso en el proyecto |
|---|---|---|
| `HHHH` | γ⁰ HH en potencia (unidades «1», relleno nan) | agua (reflexión especular), rebote doble, Δσ⁰ → Δε′ |
| `HVHV` | γ⁰ HV | vegetación (dispersión de volumen), separar vegetación inundada |
| `rtcGammaToSigmaFactor` | factor γ⁰ → σ⁰ | Dubois está escrito para σ⁰ |
| `numberOfLooks` | looks por píxel | **ENL del filtro de Lee** (cuánto moteado tiene cada píxel) |
| `mask` | 0 inválido, 1–5 sub-swath válido, 255 relleno | **máscara de validez** |
| `metadata/radarGrid/incidenceAngle` | cubo 3-D (alto, y, x) | **θ por píxel** para Dubois y Δε′ |

> **Fuente** — R2 (capas) · R16, sec. 4.3.3 p. 20 y tabla 5-3 p. 29 (`mask`, unidades y relleno de `HHHH`) · R16 sec. 3.2.4 y tabla 5-1 (cubo de metadatos) · Estatus [V]. R16 es la Rev D (2024); la producción usa una revisión posterior que puede cambiar detalles **[?]**.

### 5.1 Archivos sintéticos para el modo DEMO

Función `lectura.demo_h5`. Escribe HDF5 con la **misma estructura** que un GCOV (rutas de R2) y valores inventados para probar el código: tierra, un cauce y una llanura que «se inunda» después del evento. **No representan la realidad.**

## 6. Indicadores por escena y por par

Código: `nisar_cn/indicadores.py`.

| # | Indicador | Cálculo | Fuente |
|---|---|---|---|
| 1 | Agua abierta | HH (dB) < umbral de Otsu, acotado a [−22, −14] dB | Otsu: R10 [?] · física: R7 [V] · **acotación: R15 [?]** |
| 2 | Vegetación inundada | HH > −3.5 dB y HH − HV > 7 dB | Física: R6, R7 [V] · **umbrales: R15 [?]** |
| 3 | Cambio de agua | agua_después ∧ ¬agua_antes (ganada) y viceversa (perdida) | R15 [V] |
| 4 | Δσ⁰_HH (dB) | 10·log₁₀(σ⁰₂/σ⁰₁) en píxeles de tierra estable (no agua en ninguna fecha) | Definición; σ⁰ = γ⁰·factor (R2) [V] |
| 5 | Δε' (constante dieléctrica) | Δσ⁰_dB / (0.28·tan θ), suponiendo ks constante | Sección 2.3 [A] |
| 6 | HV/HH | cociente en dB (HV_dB − HH_dB) | Indicador de volumen frente a superficie: R6 [V]; la lectura cuantitativa es [A] |
| 7 | Coherencia | capa de coherencia del GUNW del par de 12 días | R4 [V] |
| 8 | Humedad del suelo | capas SME2 (DSG, PMI, TSR) | R3 [V] |

> **[?] Hueco — umbrales en dB.** −22, −14, −3.5 y 7 dB vienen del código del equipo (R15, `classify`) y el README no cita su origen. No encontré una referencia que los respalde para banda L: R7 no da umbrales en dB. **Pido al equipo la referencia** (autor, título, año, tabla o figura) o, si no la hay, presentarlos como parámetros calibrados a ojo sobre el Ucayali.
>
> **[?] Hueco — viento sobre el agua.** El agua agitada por viento o lluvia podría verse más brillante y escaparse del umbral. R7 no lo trata y no tengo una fuente abierta para afirmarlo. Pido referencia antes de usarlo en la presentación.
>
> **Filtro de speckle:** filtro de Lee con ENL = `numberOfLooks` (sección 5.2). La mediana 3×3 de R15 queda como opción (`clasificar(..., filtrar=True)`).

> **Cambio respecto de la versión anterior:** la clasificación ahora usa γ⁰ ya limpio (máscara + Lee) en lugar de la mediana 3×3, y Δε′ usa **θ por píxel**. Se agregan `estructuras` (rebote doble permanente), `suelo más húmedo/seco` (Δσ⁰ ≥ +2 / ≤ −2 dB con mediana 5×5, **[A][?]**) y el control de calidad.

In [ ]:
_, _, TABLA, MAPAS = pipeline.ejecutar(MODO, CACHE)
if MODO == "DEMO":
    print("⚠ MODO DEMO: valores de archivos SINTÉTICOS. No son resultados.")
cols = ["evento", "zona", "geom", "antes", "despues", "mezcla_madurez", "agua_antes_km2", "agua_despues_km2",
        "agua_ganada_km2", "dsigma_tierra_med_db", "deps_tierra_med", "hvhh_antes_db", "hvhh_despues_db", "veg_inundada_nueva_km2", "estructuras_km2", "suelo_humedecido_km2",
        "mask_valida_antes", "cobertura", "theta_med_deg", "theta"]
TABLA[cols].round({c: 3 for c in cols if c not in ("antes", "despues", "theta", "evento", "zona", "geom", "mezcla_madurez")}) if len(TABLA) else "Sin pares analizables"

### 5.2 Limpieza de los datos, paso a paso

Código: `nisar_cn/limpieza.py` y `indicadores.preparar_fecha`.

1. **Máscara de validez.** Se conservan los píxeles con `mask` entre 1 y 5 y `HHHH` > 0 finito. R17 avisa que la máscara está desalineada y deja artefactos en los bordes; por eso se erosiona 2 píxeles.
   > **Fuente** — R16 [V] · R17 [V] · erosión de 2 px: **[A]**
2. **Moteado (speckle).** Filtro de Lee en potencia lineal. La varianza relativa del moteado es 1/ENL, y **el ENL se toma de `numberOfLooks` de cada píxel**: así el filtro suaviza más donde el producto promedió menos muestras. Sustituye a la mediana 3×3 del código original del equipo.
   > **Fuente** — Fórmula de Lee (1980) **[?]** (artículo no abierto) · uso de un filtro de la familia Lee antes de umbralizar en el río Tumbes: R18 [V] · ENL = looks: **[A]**
3. **γ⁰ → σ⁰** con `rtcGammaToSigmaFactor` para todo lo que usa Dubois. La clasificación de agua usa γ⁰.
   > **Fuente** — R2 [V]
4. **Ángulo de incidencia por píxel.** Se toma la capa del cubo a la altura más cercana a 0 m y se interpola bilinealmente a la ventana. Si falla, se usa 40° y se avisa.
   > **Fuente** — R16 [V] · elección de la altura e interpolación: **[A]**
5. **Ciudad y estructuras.** El rebote doble que ya estaba en la fecha de referencia se marca como estructura y no cuenta como inundación.
   > **Fuente** — física del rebote doble: R6, R7 [V] · regla: **[A]**
6. **Agua permanente** (sección 7): agua en ≥ 80 % de las fechas anteriores al primer evento. Saca de la cuenta ríos, mar, langostineras y falsos positivos estables (arena lisa).
   > **Fuente** — separar agua permanente con la serie: R18 [V] · 80 %: **[A][?]**
7. **Control de calidad por fecha:** si más del 60 % de la ventana sale como agua, la fecha se marca dudosa y no se usa para corroborar. **[A]**
8. **Salto de madurez:** R3 advierte que mezclar PROVISIONAL y VALIDATED puede dar diferencias de procesamiento. La sección 7.2 lo mide en tierra estable. **[V]** advertencia · medición **[A]**

In [ ]:
from nisar_cn import limpieza
if len(TABLA):
    m0, r0 = MAPAS[0], TABLA.iloc[0]
    a = m0["res"]["a"]
    print(f"Par de ejemplo: {r0['evento']} · {r0['zona']} · {r0['geom']} · antes {str(r0['antes'])[:10]}")
    print("Limpieza de la fecha 'antes':", {k: (round(v, 3) if isinstance(v, float) else v) for k, v in a["limpieza"].items()})
    print("θ:", r0["theta"])
    fig, ax = plt.subplots(2, 4, figsize=(18, 8))
    vm = dict(vmin=-25, vmax=0, cmap="gray")
    ax[0, 0].imshow(a["hh_db_bruto"], **vm); ax[0, 0].set_title("1. γ⁰ HH en bruto (dB)")
    mk = a.get("mask")
    if mk is not None:
        im = ax[0, 1].imshow(mk, cmap="tab10", vmin=0, vmax=9); ax[0, 1].set_title("2. capa mask (0 inválido, 1–5 sub-swath)")
    ax[0, 2].imshow(a["valido"], cmap="Greys_r"); ax[0, 2].set_title(f"3. válido tras mask + erosión ({a['limpieza']['frac_valida_final']:.1%})")
    ax[0, 3].imshow(a["hh_db"], **vm); ax[0, 3].set_title("4. tras filtro de Lee (ENL = looks)")
    th = m0["theta"]
    if np.ndim(th):
        im = ax[1, 0].imshow(np.degrees(th), cmap="viridis"); plt.colorbar(im, ax=ax[1, 0]); ax[1, 0].set_title("5. θ por píxel (°) desde el cubo")
    else:
        ax[1, 0].text(.5, .5, f"θ constante = {np.degrees(th):.0f}°\n(sin cubo)", ha="center"); ax[1, 0].set_title("5. θ")
    lk = a.get("numberOfLooks")
    if lk is not None:
        im = ax[1, 1].imshow(lk, cmap="magma"); plt.colorbar(im, ax=ax[1, 1]); ax[1, 1].set_title("numberOfLooks (ENL del filtro)")
    cmap = plt.matplotlib.colors.ListedColormap(["#d9d2c3", "#2f8cff", "#2fd1a0"])
    ax[1, 2].imshow(np.where(a["cls"] == 255, np.nan, a["cls"]), cmap=cmap, vmin=0, vmax=2); ax[1, 2].set_title("6. clases: tierra / agua / rebote doble")
    b_ = np.isfinite(a["hh_db_bruto"]); f_ = np.isfinite(a["hh_db"])
    ax[1, 3].hist(a["hh_db_bruto"][b_].ravel(), 120, range=(-35, 5), alpha=.5, label="bruto")
    ax[1, 3].hist(a["hh_db"][f_].ravel(), 120, range=(-35, 5), alpha=.6, label="limpio (Lee)")
    ax[1, 3].axvline(a["thr"], color="r", ls="--", label=f"umbral Otsu {a['thr']:.1f} dB"); ax[1, 3].legend(); ax[1, 3].set_title("7. histograma γ⁰ HH")
    for x_ in ax.ravel()[:7]: x_.axis("off")
    plt.suptitle("Limpieza paso a paso" + ("  —  DEMO SINTÉTICO" if MODO == "DEMO" else "")); plt.tight_layout(); plt.show()
    # Efecto del filtro sobre la dispersión del moteado en tierra
    t_ = a["cls"] == 0
    print(f"Desviación estándar de γ⁰ HH en tierra: bruto {np.nanstd(a['hh_db_bruto'][t_]):.2f} dB → limpio {np.nanstd(a['hh_db'][t_]):.2f} dB")

### 5.3 Eficiencia: cuánto se lee de cada escena

Cada GCOV pesa 6–7 GB (R14). El código abre el archivo remoto una vez, lee solo los recortes de las zonas que lo usan (bloques de 4 MB guardados en memoria) y guarda cada recorte en `cache/` como `.npz` comprimido. La segunda corrida no vuelve a NASA. La tabla compara lo guardado con el tamaño de la escena.

> **Fuente** — streaming: ASF «Stream Data» [V] · tamaño de escena: R14 [V] · estrategia de bloques y caché: **[A]**

In [ ]:
import os
filas = []
for f in sorted(CACHE.glob("*.npz")):
    z = np.load(f)
    px = z["c_HHHH"].size if "c_HHHH" in z else 0
    filas.append(dict(recorte=f.name[:70] + "…", pixeles=px, capas=sum(k.startswith("c_") for k in z.files),
                      MB_en_disco=round(os.path.getsize(f) / 1e6, 2)))
EF = pd.DataFrame(filas)
if len(EF):
    tot = EF["MB_en_disco"].sum()
    print(f"{len(EF)} recortes guardados, {tot:.0f} MB en total. Una sola escena completa pesa ~6 800 MB (R14).")
    print(f"Media por recorte: {EF['MB_en_disco'].mean():.1f} MB → ~{100 * EF['MB_en_disco'].mean() / 6800:.2f} % de una escena.")
EF.head(10)

### 6.1 Mapas del primer par: clases, Δσ⁰, Δε′ e impacto

In [ ]:
if len(TABLA):
    r, m = TABLA.iloc[0], MAPAS[0]
    res, dsig, deps, tierra, imp = m["res"], m["dsig"], m["deps"], m["tierra"], m["impacto"]
    from nisar_cn.mapas import COLOR_IMPACTO
    cm_imp = plt.matplotlib.colors.ListedColormap([[1, 1, 1, 0]] + [np.array(COLOR_IMPACTO[k][:3]) / 255 for k in range(1, 8)])
    fig, ax = plt.subplots(1, 5, figsize=(22, 4.5))
    cmap = plt.matplotlib.colors.ListedColormap(["#d9d2c3", "#2f8cff", "#2fd1a0"])
    for i, k in enumerate(("a", "d")):
        ax[i].imshow(np.where(res[k]["cls"] == 255, np.nan, res[k]["cls"]), cmap=cmap, vmin=0, vmax=2)
        ax[i].set_title(("Antes " if k == "a" else "Después ") + str(r["antes" if k == "a" else "despues"])[:10])
    im = ax[2].imshow(np.where(tierra, dsig, np.nan), cmap="RdBu_r", vmin=-4, vmax=4); ax[2].set_title("Δσ⁰ HH (dB), tierra estable"); plt.colorbar(im, ax=ax[2])
    im = ax[3].imshow(np.where(tierra, deps, np.nan), cmap="BrBG", vmin=-15, vmax=15); ax[3].set_title("Δε′ con θ por píxel [A]"); plt.colorbar(im, ax=ax[3])
    ax[4].imshow(res["d"]["hh_db"], cmap="gray", vmin=-25, vmax=0); ax[4].imshow(np.where((imp == 0) | (imp == 255), np.nan, imp), cmap=cm_imp, vmin=0, vmax=7)
    ax[4].set_title("Impacto: azul perm. · cian nueva · naranja retirada\nverde veg. inund. · lila suelo húmedo · gris ciudad")
    for a_ in ax: a_.axis("off")
    plt.suptitle(f"{r['evento']} · {config.ZONAS[r['zona']]['nombre']} · {r['geom']}" + ("  —  DEMO SINTÉTICO" if MODO == "DEMO" else ""))
    plt.tight_layout(); plt.show()

## 7. Serie temporal: todas las fechas, agua permanente, calidad y madurez

Código: `nisar_cn/serie.py`. Para cada zona con evento y cada órbita (track, dirección, frame) con al menos dos fechas fuera del corte de datos (R11), cada fecha se compara con la primera. Se escriben capas por fecha y un GIF animado por serie para el mapa.

> **Fuente** — misma geometría para que las fechas sean comparables: R15 [V] · corte de datos: R11 [V] · agua permanente con la serie: R18 [V] (umbral 80 %: **[A][?]**)

In [ ]:
from nisar_cn import serie as serie_mod
SERIE = serie_mod.serie_temporal(CAT, MODO, CACHE, SALIDAS)
pd.DataFrame([{"zona": s["zona"], "órbita": s["geom"], "fechas": len(s["frames"]),
               "desde": s["frames"][0]["fecha"][:10], "hasta": s["frames"][-1]["fecha"][:10],
               "agua permanente (km²)": s.get("agua_permanente_km2"), "fechas previas usadas": s.get("fechas_previas_usadas"),
               "fechas dudosas": sum(f.get("calidad", {}).get("sospechosa", False) for f in s["frames"])} for s in SERIE])

### 7.1 Evolución de cada serie

In [ ]:
# Evolución por serie: agua, agua nueva vs permanente, suelo más húmedo, con eventos
import datetime as dt
n = len(SERIE)
if n:
    fig, axs = plt.subplots((n + 1) // 2, 2, figsize=(16, 3.2 * ((n + 1) // 2)), squeeze=False)
    for s, ax in zip(SERIE, axs.ravel()):
        t = [dt.datetime.fromisoformat(f["fecha"]) for f in s["frames"]]
        g = lambda k: [f["metricas"].get(k) for f in s["frames"]]
        ax.plot(t, g("agua_km2"), "o-", label="agua (km²)")
        if any(v is not None for v in g("agua_nueva_vs_permanente_km2")):
            ax.plot(t, g("agua_nueva_vs_permanente_km2"), "s-", label="agua no permanente (km²)")
        ax.plot(t, g("suelo_humedecido_km2"), "^-", label="suelo más húmedo (km²)")
        for f_, tt in zip(s["frames"], t):
            if f_.get("calidad", {}).get("sospechosa"):
                ax.axvline(tt, color="k", lw=4, alpha=.15)
        for e in config.EVENTOS:
            if s["zona"] in e["zonas"]:
                ax.axvline(e["inicio"], color="r", ls="--", lw=1); ax.text(e["inicio"], ax.get_ylim()[1] * .9, e["id"], color="r", fontsize=8)
        ax.set_title(f"{config.ZONAS[s['zona']]['nombre']} · {s['geom']}", fontsize=10); ax.legend(fontsize=7)
    plt.tight_layout(); plt.show()
    print("Franjas grises = fechas marcadas como dudosas por el control de calidad.")

### 7.2 ¿El cambio es del terreno o del procesamiento? (salto PROVISIONAL → VALIDATED)

R3 advierte que mezclar madureces puede producir diferencias que vienen del procesamiento y no del terreno. Se grafica la mediana de σ⁰ HH en **tierra estable** (ni agua ni rebote doble en ninguna de las dos fechas) por fecha y se colorea por madurez. Un escalón que coincide con el cambio de madurez y **no** con lluvia apunta a calibración; uno que coincide con lluvia apunta a humedad.

> **Fuente** — R3 [V] · método de comparación: **[A]**

In [ ]:
if SERIE:
    fig, ax = plt.subplots(figsize=(14, 4))
    for s in SERIE:
        t = [dt.datetime.fromisoformat(f["fecha"]) for f in s["frames"]]
        v = [f["metricas"].get("s0hh_tierra_med_db") for f in s["frames"]]
        ax.plot(t, v, "-", color="grey", alpha=.4)
        ax.scatter(t, v, c=["#5be3a0" if f["madurez"] == "VALIDATED" else "#4c78a8" for f in s["frames"]], s=30, zorder=3)
    ax.axvline(dt.datetime(2026, 10, 21, tzinfo=config.UTC), color="green", ls=":", label="inicio VALIDATED (R3)")
    ax.scatter([], [], c="#4c78a8", label="PROVISIONAL"); ax.scatter([], [], c="#5be3a0", label="VALIDATED")
    ax.set_ylabel("mediana σ⁰ HH en tierra (dB)"); ax.legend(); ax.set_title("Nivel de σ⁰ en tierra estable por fecha y madurez")
    plt.tight_layout(); plt.show()

## 8. Otros productos NISAR: humedad del suelo (SME2) y coherencia (GUNW)

Código: `nisar_cn/extra.py`. Busca en el CMR todas las escenas SME2 y GUNW de cada zona, lee la ventana por streaming y guarda el recorte.

- **SME2**: humedad del suelo (tres algoritmos: DSG, PMI, TSR) a 200 m. Se usa la capa base `soilMoisture`. Es la **medición directa** de lo que nuestro Δε′ solo aproxima: si llueve, deben subir los dos.
- **GUNW**: interferograma desenrollado de dos pasadas, con capa de coherencia, a 80 m. Coherencia baja = la superficie cambió entre pasadas (agua, lodo, cosecha).

> **Fuente** — R3 (SME2 a 200 m con DSG/PMI/TSR; SME2 sin versión VALIDATED; GUNW a 80 m con coherencia) [V] · capa base `soilMoisture`: R3 «Known Issues» [V] · las rutas exactas se **buscan por nombre** en cada archivo y se imprimen: **[?]** hasta confirmarlas con la salida real · corte de coherencia baja 0,3: **[A][?]**

In [ ]:
from nisar_cn import extra
EXTRA = extra.productos_extra(sorted({s["zona"] for s in SERIE}), CACHE, SALIDAS, MODO) if MODO == "REAL" else {}
if EXTRA:
    fig, ax = plt.subplots(1, 2, figsize=(16, 4))
    for z, d in EXTRA.items():
        for prod, a_, k, esc in (("SME2", ax[0], "humedad_med_m3m3", 1), ("GUNW", ax[1], "coherencia_med", 1)):
            fr = d.get(prod, [])
            if fr:
                a_.plot([dt.datetime.fromisoformat(f["fecha"]) for f in fr], [f["metricas"].get(k) for f in fr], "o-", label=config.ZONAS[z]["nombre"])
    for e in config.EVENTOS:
        for a_ in ax: a_.axvline(e["inicio"], color="r", ls="--", lw=.8)
    ax[0].set_title("Humedad del suelo NISAR SME2 (mediana, m³/m³)"); ax[1].set_title("Coherencia NISAR GUNW (mediana)")
    for a_ in ax: a_.legend(fontsize=7)
    plt.tight_layout(); plt.show()
else:
    print("DEMO o sin escenas: SME2/GUNW se procesan solo en modo REAL.")

## 9. Corroboración con fuentes independientes

Pregunta: **¿lo que mide NISAR coincide con lo que pasó en el terreno?** Se contrasta con evidencia que **no viene del radar**:

1. **Lluvia**: CHIRPS v3 (satélite + estaciones, abierto, R23), NASA GPM IMERG (satélite, R19), Open-Meteo (reanálisis, R21) y las estaciones de Senamhi que citan las notas, ubicadas con sus coordenadas oficiales (R22).
2. **Caudal del río Tumbes** frente a los umbrales publicados para esa cuenca (R18), con el solape real entre nuestra caja y el área del estudio.
3. **Confirmación institucional**: Carta Internacional (R20) y Copernicus EMS (R24).
4. **Crónica de noticias** con fechas y lugares (N1–N17).
5. **Puntaje** de la correspondencia con las medidas estándar de verificación (R26) y **prueba de sensibilidad** de nuestros umbrales.

Código: `nisar_cn/corroboracion.py`, `nisar_cn/lluvia.py`.

> **IMERG: causa verificada.** El 15 nov 2026, con la llave de este proyecto, GES DISC respondió **HTTP 403 «EULA Acceptance Failure»** junto con un enlace `resolution_url` de Earthdata Login (R25). La llave es válida; falta **aceptar una vez la EULA** de «NASA GESDISC DATA ARCHIVE». La celda 9.1 imprime ese enlace: ábrelo con tu cuenta, acepta y vuelve a correr.
>
> Mientras tanto, **CHIRPS v3 «sat» cubre el hueco**: reparte el total pentadal (mezclado con estaciones) entre los días usando el propio IMERG Late V07 y no pide llave (R23) · Estatus [V]


### 9.1 Lluvia: satélite, reanálisis y estaciones

In [ ]:
from nisar_cn import lluvia, corroboracion as corr
LLUVIA = lluvia.lluvia_todas(CACHE)
IMERG, CHIRPS = {}, {}
if MODO == "REAL":
    try:
        IMERG = corr.imerg_zonas(CACHE)          # {} si GES DISC no da acceso (imprime el motivo y el enlace)
    except Exception as e:
        print("IMERG no disponible:", type(e).__name__, str(e)[:200])
    try:
        CHIRPS = corr.chirps_zonas(CACHE)        # abierto, sin llave
    except Exception as e:
        print("CHIRPS no disponible:", type(e).__name__, str(e)[:200])
print("Lluvia satelital en uso:", ", ".join(k for k, v in (("IMERG", IMERG), ("CHIRPS v3", CHIRPS)) if v) or "ninguna (DEMO o sin acceso)")
zonas_g = [z for z in ("tumbes", "zarumilla", "pizarro", "piura", "chulucanas") if z in config.ZONAS]
fig, axs = plt.subplots(len(zonas_g), 1, figsize=(15, 2.6 * len(zonas_g)), sharex=True)
for z, ax in zip(zonas_g, axs):
    for fuente, d, c, w in (("IMERG (satélite)", IMERG.get(z), "#1f4fd1", .8), ("CHIRPS v3 (satélite + estaciones)", CHIRPS.get(z), "#e8a33d", .6),
                            ("Open-Meteo (reanálisis)", LLUVIA.get(z), "#9ec5ff", .4)):
        if d:
            t = [dt.datetime.fromisoformat(f) for f in d["fechas"]]
            ax.bar(t, [v or 0 for v in d["mm"]], width=w, color=c, label=fuente, alpha=.85)
    for x in corr.ESTACIONES:
        if x["zona"] == z:
            ax.scatter(x["fin"], x["mm"], marker="*", s=160, c="#e45756", zorder=5)
            ax.annotate(f"{x['estacion']} {x['mm']:.0f} mm ({x['fuente']})", (x["fin"], x["mm"]), fontsize=7, xytext=(4, 2), textcoords="offset points")
    for s in SERIE:
        if s["zona"] == z:
            for f in s["frames"]:
                ax.axvline(dt.datetime.fromisoformat(f["fecha"]), color="k", lw=.6, alpha=.5)
    for e in config.EVENTOS:
        if z in e["zonas"]:
            ax.axvline(e["inicio"], color="r", ls="--", lw=1)
    ax.set_ylabel("mm/día"); ax.set_title(config.ZONAS[z]["nombre"], fontsize=10, loc="left")
axs[0].legend(fontsize=8, loc="upper left")
plt.suptitle("Lluvia diaria: satélite, reanálisis y estaciones (★). Líneas negras = pasadas NISAR; rojas = eventos"); plt.tight_layout(); plt.show()


### 9.1 b ¿Los productos en malla ven la lluvia que midieron las estaciones?

Cada lectura de estación citada en la prensa se compara con la lluvia de cada producto **en la celda de la estación** (coordenadas del mapa de estaciones de Senamhi, R22). Si la nota no identifica una estación de Senamhi, o el producto no se extrajo en ese punto, se usa la celda de la zona (columna `*_celda`). Se dan el día de cierre y los 2 días que terminan ese día, porque una ventana de 24 h cruza la medianoche.

**Cómo leerla.** Si la fracción captada es baja, el producto en malla (5–10 km) suaviza una lluvia convectiva local. Entonces «≥ 30 mm acumulados» en 9.3 puede no dispararse aunque haya llovido fuerte en un punto. Por eso la regla de 9.3 también acepta estaciones y noticias como evidencia, y 9.3 b mide cuánto depende el veredicto de ese umbral.

> **Fuente** — lecturas: N1, N3, N5, N9, N10, N15 [V] · coordenadas: R22 (leídas el 15 nov 2026) [V] · ventana de 1–2 días: **[A]**. CHIRPS etiqueta los días en UTC (R23) y Open-Meteo en hora de Lima (R21); el desfase de 5 h es otra razón para mirar 2 días.
>
> **El Salto, 2 nov:** N3 da 104,1 mm «en 24 horas» y N5 da 110,3 mm «durante la madrugada» sin periodo. Son dos ventanas distintas y se muestran ambas [V].
>
> **[?] Datos en bruto de Senamhi:** las tablas existen (El Salto 103041, Cabo Inga 103043, Puerto Pizarro 103038; al 15 nov 2026 el último mes disponible es octubre). Están detrás de un CAPTCHA (Cloudflare Turnstile), así que no se descargan de forma automática. Si el equipo las baja a mano, reemplazan los valores de prensa.


In [ ]:
if MODO == "REAL":
    EST = corr.contraste_estaciones({"chirps": CHIRPS, "imerg": IMERG, "reanalisis": LLUVIA})
    display(EST)
    c2 = [c for c in EST.columns if c.endswith("_2dias") and EST[c].notna().any()]
    print("Fracción de la lluvia de estación que capta cada producto (ventana de 2 días, mediana):",
          {c[:-6]: round(float((EST[c] / EST["mm_estacion"]).median()), 2) for c in c2})
else:
    print("DEMO: el contraste con estaciones solo se calcula en modo REAL.")


### 9.2 Caudal del río Tumbes frente a los umbrales de inundación

Breña Aliaga et al. (2026) calibraron con Sentinel-1 y reportes de INDECI que en el río Tumbes **los impactos significativos empiezan en 743,49 m³/s** (151 ha inundadas, 157 habitantes afectados) y escalan linealmente hasta **1 629,02 m³/s** (1 234 ha agrícolas, 749 habitantes). Senamhi reportó **1 021,33 m³/s en El Tigre el 2 nov a las 5:00**, por encima del umbral rojo de 1 000 m³/s. Según ese estudio, ese caudal implica inundación en la llanura del Tumbes: **la primera pasada NISAR posterior al 2 nov sobre Tumbes debería mostrar agua nueva**.

La celda siguiente comprueba dos cosas que antes eran supuestos:
1. **¿Nuestras cajas caen en el área del estudio?** Solape con el marco «Study Area» de la fig. 2 de R18 (UTM 17S, del El Tigre al mar).
2. **¿Ya hay una pasada NISAR posterior al 2 nov?** Consulta en vivo al CMR, en todas las colecciones y madureces.

> **Fuente** — R18 (resumen, «Highlights», sec. 2.1, 2.2.2 y fig. 2; releído el 15 nov 2026) [V] · N3, N4, N5 [V] · área de estudio leída de los ejes de la fig. 2, ±1 km **[A]**. La inferencia «debería verse agua nueva» es **[A]**: R18 usa banda C y **solo VV** (sec. 2.2.2); nosotros usamos banda L con HH/HV.


In [ ]:
SOLAPE = pd.DataFrame([corr.solape_r18(z) for z in ("tumbes", "pizarro", "zarumilla")])
display(SOLAPE)
POST = None
if MODO == "REAL":
    POST = corr.escenas_despues()
    print(f"Escenas NISAR posteriores al 2 nov 12:00 UTC (CMR, consulta del {dt.date.today()}): {len(POST)}")
    if POST:
        display(pd.DataFrame(POST))
    else:
        print("→ todavía ninguna: la predicción de 9.2 queda pendiente (pausa de producción, R12).")
fig, ax1 = plt.subplots(figsize=(14, 4))
for s in SERIE:
    if s["zona"] in ("tumbes", "pizarro"):
        t = [dt.datetime.fromisoformat(f["fecha"]) for f in s["frames"]]
        ax1.plot(t, [f["metricas"].get("agua_nueva_vs_permanente_km2", f["metricas"].get("agua_nueva_vs_base_km2")) for f in s["frames"]],
                 "o-", label=f"NISAR agua no permanente · {config.ZONAS[s['zona']]['nombre']} {s['geom']}")
ax1.set_ylabel("km² (NISAR)")
ax2 = ax1.twinx()
for q in corr.CAUDALES_TUMBES:
    ax2.scatter(q["fecha"], q["m3s"], marker="D", s=90, c="#e45756", zorder=5, label=f"caudal El Tigre ({q['fuente']})")
ax2.axhline(corr.UMBRAL_IMPACTO_BRENA_M3S, color="orange", ls="--", label="inicio de impactos 743,49 m³/s (R18)")
ax2.axhline(corr.UMBRAL_ALERTA_ROJA_M3S, color="red", ls=":", label="alerta roja 1 000 m³/s (N3, N5)")
ax2.axhline(corr.UMBRAL_EXTREMO_BRENA_M3S, color="darkred", ls="--", label="pico extremo 1 629 m³/s (R18)")
ax2.set_ylabel("m³/s"); ax2.set_ylim(0, 1800)
h1, l1 = ax1.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, fontsize=7, loc="upper left"); ax1.set_title("Río Tumbes: caudal observado y umbrales vs agua medida por NISAR")
plt.tight_layout(); plt.show()

### 9.3 Matriz de correspondencia NISAR ↔ evidencia en tierra

Para cada periodo entre dos pasadas NISAR consecutivas (por zona y órbita) se cruza:
- **evidencia en tierra**: noticia o evento en el periodo, estación con lluvia reportada, o lluvia acumulada ≥ 30 mm (IMERG, CHIRPS o reanálisis);
- **señal NISAR**: +0,5 km² de agua, +0,5 km² de vegetación inundada nueva o +1 km² de suelo más húmedo.

Veredictos: **coinciden** · **NISAR no lo ve** (hubo evidencia y el radar no muestra cambio: agua ya drenada, inundación urbana bajo el umbral o fuera de la caja) · **NISAR sin noticia** (cambio sin evidencia: cultivos, riego, calibración) · **dato dudoso**.

**9.3 b Puntaje y sensibilidad.** La matriz se trata como un pronóstico dicotómico: «sí» de NISAR = señal, «sí» observado = evidencia en tierra. Con la tabla de contingencia se calculan POD = aciertos/(aciertos+fallos), FAR = falsas alarmas/(aciertos+falsas alarmas), CSI = aciertos/(aciertos+fallos+falsas alarmas) y el sesgo de frecuencia (R26); los periodos con dato dudoso se excluyen. Luego se repite con **27 combinaciones de umbrales** (lluvia 15/30/50 mm, agua 0,25/0,5/1 km², suelo 0,5/1/2 km²) para medir cuánto dependen las conclusiones de nuestros parámetros. Si el veredicto casi no cambia, la elección del umbral no decide la conclusión; si cambia mucho, hay que decirlo en la presentación.

> **Fuente** — regla y umbrales: **[A][?]** (los mismos del mapa; ahora con su sensibilidad medida) · fórmulas: R26 [V] · insumos: R18–R23, N1–N17 [V]

In [ ]:
MC = corr.matriz_correspondencia(SERIE, LLUVIA, IMERG, CHIRPS)
if len(MC):
    colores = {"coinciden": "#2ca02c", "NISAR no lo ve": "#d62728", "NISAR sin noticia": "#ff7f0e", "dato dudoso": "#7f7f7f", "sin cambio ni noticia": "#c7c7c7"}
    MC["serie"] = MC["zona"].map(lambda z: config.ZONAS[z]["nombre"]) + " · " + MC["geom"]
    ys = {s: i for i, s in enumerate(MC["serie"].unique())}
    fig, ax = plt.subplots(figsize=(15, 0.5 * len(ys) + 1.5))
    for _, r in MC.iterrows():
        ax.plot([r["desde"], r["hasta"]], [ys[r["serie"]]] * 2, lw=9, color=colores[r["veredicto"]], solid_capstyle="butt")
    ax.set_yticks(list(ys.values())); ax.set_yticklabels(list(ys.keys()), fontsize=8)
    for k, c in colores.items(): ax.plot([], [], lw=8, color=c, label=k)
    ax.legend(ncol=5, fontsize=8, loc="lower left", bbox_to_anchor=(0, 1.01)); ax.set_title("Correspondencia por periodo entre pasadas", pad=34)
    plt.tight_layout(); plt.show()
    display(MC["veredicto"].value_counts().to_frame("periodos"))
    display(MC.drop(columns=["serie"]))

In [ ]:
if len(MC):
    PUNT = corr.puntajes(MC)
    display(pd.DataFrame([PUNT]))
    SENS = corr.sensibilidad(SERIE, LLUVIA, IMERG, CHIRPS)
    num = SENS[["CSI", "POD", "FAR"]].apply(pd.to_numeric)
    print(f"Veredicto igual al de la regla base: mediana {SENS['igual_a_base'].median():.0%}, mínimo {SENS['igual_a_base'].min():.0%} (27 combinaciones)")
    print(f"CSI {num['CSI'].min():.2f}–{num['CSI'].max():.2f} · POD {num['POD'].min():.2f}–{num['POD'].max():.2f} · FAR {num['FAR'].min():.2f}–{num['FAR'].max():.2f}")
    display(SENS.sort_values("igual_a_base"))
else:
    SENS = pd.DataFrame()


### 9.4 Confirmación institucional

> **Carta Internacional «Espacio y Grandes Catástrofes», activación 1058** (3 nov 2026, 14:52 UTC+2). Pedida por INDECI en nombre de CONIDA por el desborde del río Zarumilla sobre Puerto Perú (Aguas Verdes): más de 150 viviendas dañadas y más de 700 personas afectadas. Gestor del proyecto: CONIDA. Releída el 15 nov 2026: **todavía no lista productos**. Cuando CONIDA publique mapas, se comparan con la capa «inundación nueva» de NISAR en Zarumilla. — R20 [V]
>
> **Copernicus EMS Rapid Mapping:** la celda consulta la API pública del portal. El 15 nov 2026 devolvió 268 activaciones (la última, EMSR935, del 19 oct 2026) y **ninguna para Perú**. Hueco cerrado: Copernicus no tiene activación para este evento. — R24 [V]
>
> **Ojo:** la caja de Zarumilla cae fuera del área de R18 (9.2). Para Zarumilla, la confirmación independiente es la Carta 1058 y las noticias, no los umbrales del río Tumbes.


In [ ]:
CEMS = None
if MODO == "REAL":
    try:
        CEMS = corr.cems_peru()
        print(f"Copernicus EMS: {CEMS['total']} activaciones, la más reciente {CEMS['ultima']}; para Perú: {len(CEMS['peru'])}")
        for x in CEMS["peru"]:
            print("  ", x["code"], x["activationTime"], x["name"])
    except Exception as e:
        print("Copernicus EMS no disponible:", type(e).__name__)
print(corr.CARTA["texto"])


### 9.5 Resumen por zona

In [ ]:
# Resumen por zona: qué dice NISAR y qué dicen las otras fuentes
filas = []
for z in sorted({s["zona"] for s in SERIE}):
    ss = [s for s in SERIE if s["zona"] == z]
    agua_max = max((f["metricas"].get("agua_nueva_vs_permanente_km2") or f["metricas"].get("agua_nueva_vs_base_km2") or 0) for s in ss for f in s["frames"])
    mc = MC[MC["zona"] == z] if len(MC) else MC
    filas.append(dict(zona=config.ZONAS[z]["nombre"], series=len(ss),
                      NISAR_max_agua_no_perm_km2=round(agua_max, 2),
                      periodos_coinciden=int((mc["veredicto"] == "coinciden").sum()) if len(mc) else 0,
                      periodos_no_lo_ve=int((mc["veredicto"] == "NISAR no lo ve").sum()) if len(mc) else 0,
                      noticias=len([c for c in config.CRONICA if z in c["zonas"]]),
                      estaciones=", ".join(f"{x['estacion']} {x['mm']:.0f} mm" for x in corr.ESTACIONES if x["zona"] == z),
                      carta_1058="sí" if z == corr.CARTA["zona"] else ""))
pd.DataFrame(filas)

## 10. Tabla para el storytelling, manifiesto y mapa

Código: `nisar_cn/exportar.py`. Une cada resultado con su evento, su titular y el enlace a la noticia. El manifiesto mantiene la forma `window.NISAR_MANIFEST` que lee `app.js` del equipo y agrega los campos `zona` y `evento`.

Además escribe las capas PNG de cada par (`nisar_cn/mapas.py`) y abre el mapa `2_mapa/MAPA_NISAR.html`, que muestra todas las zonas, los eventos con su noticia y, para cada par, SAR antes/después, agua, cambio, Δσ⁰ y Δε′.

> **Fuente** — R15 (formato del manifiesto) · N1–N6 · Estatus [V]

También procesa la **serie temporal** (`nisar_cn/serie.py`): todas las fechas NISAR disponibles por zona y órbita, cada una comparada con la primera fecha. Escribe las capas para la barra de tiempo del mapa y un **GIF animado** por serie en `salidas/serie_<modo>/`.

Para contrastar, agrega la **lluvia diaria** de cada zona (Open-Meteo, reanálisis ECMWF; `nisar_cn/lluvia.py`), la **humedad del suelo NISAR L3 SME2** y la **coherencia NISAR L2 GUNW** (`nisar_cn/extra.py`), y la **crónica de noticias y reportes oficiales** fechados (`config.CRONICA`).

In [ ]:
STORY = exportar.storytelling(TABLA, MODO) if len(TABLA) else pd.DataFrame()
CAPAS = exportar.capas_mapa(TABLA, MAPAS, MODO, SALIDAS) if len(TABLA) else []
import json
csv, js = exportar.escribir(STORY, MODO, SALIDAS, CAPAS, SERIE, LLUVIA, EXTRA, IMERG or CHIRPS,
                            json.loads(MC.drop(columns=["serie"], errors="ignore").to_json(orient="records", date_format="iso")) if len(MC) else [])
MC.to_csv(SALIDAS / f"correspondencia_{MODO.lower()}.csv", index=False)
print("Escrito:", csv.name, js.name, f"· {len(CAPAS)} pares · {len(SERIE)} series con GIF · correspondencia_{MODO.lower()}.csv")
VISOR = (RAIZ / "2_mapa" / "MAPA_NISAR.html").resolve()
print("Mapa:", VISOR)
import webbrowser; webbrowser.open(VISOR.as_uri())

## 11. Estado de los huecos

La tabla siguiente se calcula con los datos de esta corrida: indica qué hueco quedó cerrado con evidencia y cuál sigue abierto.

In [ ]:
def estado(cond_cerrado, txt_ok, txt_no):
    return ("cerrado", txt_ok) if cond_cerrado else ("abierto", txt_no)
real = MODO == "REAL"
th_ok = bool(len(TABLA)) and TABLA["theta"].str.startswith("θ interpolado").any()
mask_ok = bool(len(TABLA)) and TABLA["mask_valida_antes"].notna().any()
sme_ok = any(d.get("SME2") for d in EXTRA.values()) if EXTRA else False
gun_ok = any(d.get("GUNW") for d in EXTRA.values()) if EXTRA else False
frac_tumbes = float(SOLAPE.set_index("zona").loc["tumbes", "fraccion"])
sens_ok = len(SENS) > 0
H = [
    ("Valores de `mask`", *estado(mask_ok, "interpretada con R16 (0 / 1–5 / 255)", "no se leyó la capa")),
    ("θ por píxel", *estado(th_ok, "interpolado del cubo radarGrid (R16)", "se usó 40°: ver la columna 'theta'")),
    ("Rutas SME2", *estado(sme_ok, "leídas en archivos reales", "sin escenas o sin ruta: ver la salida de la sección 8")),
    ("Rutas GUNW (coherencia)", *estado(gun_ok, "leídas en archivos reales", "sin escenas o sin ruta: ver la sección 8")),
    ("9.1 Lluvia satelital", *estado(bool(IMERG or CHIRPS), "IMERG integrado" if IMERG else "CHIRPS v3 «sat» integrado (reparte con IMERG Late V07, R23)", "sin lluvia satelital (DEMO o sin acceso)")),
    ("9.1 IMERG directo desde GES DISC", *estado(bool(IMERG), "leído con la llave", "403 «EULA Acceptance Failure»: aceptar la EULA con el enlace que imprime 9.1 (R25)")),
    ("9.1 b Estación vs malla", *estado(real, "medido: ver la tabla 9.1 b", "solo en modo REAL")),
    ("9.1 b El Salto 2 nov: 104,1 vs 110,3 mm", "cerrado", "dos ventanas distintas: N3 24 h, N5 madrugada sin periodo [V]"),
    ("9.1 b Estaciones Senamhi en bruto", "abierto", "tablas detrás de un CAPTCHA (R22); coordenadas verificadas, valores de prensa"),
    ("9.2 Caja de Tumbes dentro del área de R18", *estado(frac_tumbes >= 0.9, f"{frac_tumbes:.0%} de la caja dentro (fig. 2 de R18)", f"solo {frac_tumbes:.0%} dentro")),
    ("9.2 Banda C (R18) vs banda L (NISAR)", "abierto", "R18 usa Sentinel-1 VV; trasladar sus umbrales a NISAR HH es [A]"),
    ("9.2 Escenas posteriores al 2 nov", *((estado(bool(POST), f"{len(POST)} escenas en el CMR", f"ninguna en el CMR al {dt.date.today()}: pausa de producción (R12)")) if real else ("—", "solo en modo REAL"))),
    ("9.3 Umbrales de la regla (30 mm / 0,5 km² / 1 km²)", "medido" if sens_ok else "abierto",
     (f"sensibilidad 9.3 b: veredicto igual a la regla base en {SENS['igual_a_base'].median():.0%} de los periodos (mediana de 27 combinaciones); sin referencia externa [A]") if sens_ok else "sin matriz"),
    ("9.4 Copernicus EMS", *estado(CEMS is not None, f"consultado: {len(CEMS['peru']) if CEMS else 0} activaciones para Perú (R24)", "no consultado (DEMO o sin red)")),
    ("9.4 Carta 1058: mapas de CONIDA", "abierto", "sin productos publicados al 15 nov 2026 (R20)"),
    ("Umbrales de agua en dB (−22/−14; −3,5/7)", "abierto", "sin referencia para banda L [?]; R18 da método (Bmax Otsu + HAND) pero sus valores son de banda C"),
    ("Parámetros ±2 dB, 80 %, 60 %, coherencia 0,3", "abierto", "parámetros [A] sin referencia [?]"),
    ("Fórmula del filtro de Lee", "abierto", "Lee (1980) no abierto [?]"),
    ("Otsu (1979), Dubois (1995), Topp (1980) originales", "abierto", "solo fuentes secundarias"),
    ("Salto PROVISIONAL → VALIDATED", "medido, no corregido", "ver sección 7.2"),
    ("Máscara de agua permanente externa (JRC) y HAND", "abierto", "páginas del dataset no accesibles [?]; se usa la serie propia"),
]
pd.DataFrame(H, columns=["hueco", "estado", "detalle"])
